# 강남노인종합복지관 실무형 크롤링 파이프라인

이 Notebook은 **운영용 코드를 한 번에 실행하는 용도보다 각 단계가 실제로 어떻게 동작하는지 검증하는 용도**입니다.

운영 실행은 프로젝트 루트의 `main.py`를 사용합니다.

## 파이프라인
`목록 수집 → 목록 파싱 → 상세 수집 → 전처리/검증 → 증분 상태 저장 → MySQL UPSERT`

In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_collection_pipeline.config import settings

print("프로젝트:", settings.project_dir)
print("수집 페이지:", settings.start_page, "~", settings.end_page)
print("DB 사용:", settings.db_enabled)

프로젝트: D:\AI\data_analytics\crawling\gangnam_senior_program_pipeline_v2.0\gangnam_senior_program_pipeline_pro
수집 페이지: 1 ~ 15
DB 사용: False


## 1. HTTP 재시도 설정 확인

In [2]:
from src.data_collection_pipeline.http_client import build_session

session = build_session()
print(session.headers["User-Agent"])
print("HTTP Session 생성 완료")

Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36
HTTP Session 생성 완료


## 2. 목록 1페이지만 테스트 수집

In [3]:
from src.data_collection_pipeline.crawling import run_crawling

raw_list_dir = run_crawling(
    start_page=1,
    end_page=1,
)

raw_list_dir

2026-08-27 16:10:49,762 | INFO | src.data_collection_pipeline.crawling | 목록 수집 시작 | page=1~1 | batch=20260827_161049
2026-08-27 16:10:49,900 | INFO | src.data_collection_pipeline.crawling | 목록 수집 성공 | page=1 | status=200 | bytes=142264 | gangnam_notice_page_001.html
2026-08-27 16:10:49,901 | INFO | src.data_collection_pipeline.crawling | 목록 수집 종료 | 성공=1 | 실패=0 | batch=20260827_161049


WindowsPath('D:/AI/data_analytics/crawling/gangnam_senior_program_pipeline_v2.0/gangnam_senior_program_pipeline_pro/data/raw/list/20260827_161049')

## 3. 목록 HTML 파싱

In [4]:
from src.data_collection_pipeline.extract import run_extract

list_csv = run_extract(raw_list_dir)
list_df = pd.read_csv(list_csv, encoding="utf-8-sig")

print("목록 건수:", len(list_df))
display(list_df.head(10))

2026-08-27 16:10:52,619 | INFO | src.data_collection_pipeline.extract | 목록 파싱 성공 | gangnam_notice_page_001.html | 17건
2026-08-27 16:10:52,625 | INFO | src.data_collection_pipeline.extract | 목록 파싱 종료 | raw=1개 | unique=17건 | 실패=0 | D:\AI\data_analytics\crawling\gangnam_senior_program_pipeline_v2.0\gangnam_senior_program_pipeline_pro\data\interim\20260827_161049\notice_list.csv


목록 건수: 17


,source_page,notice_no,notice_id,category,title,views,post_date,detail_url
0,1,공지,1121,NaN,무더위쉼터 특별운영안내(8월 한달간 주말및 공휴일 개방운영),1958,2026-07-27,https://www.gangnam.go.kr/office/gnsw/board/gn...
1,1,공지,1117,평생교육,[평생교육] 2026년 평생교육 하계방학 안내,534,2026-07-20,https://www.gangnam.go.kr/office/gnsw/board/gn...
2,1,공지,1095,홍보,[홍보] 홈페이지 웹버전 확대기능 가능 안내,438,2026-05-27,https://www.gangnam.go.kr/office/gnsw/board/gn...
3,1,공지,1074,통합돌봄,[통합돌봄] 강남구 통합돌봄 신청 안내,1476,2026-04-13,https://www.gangnam.go.kr/office/gnsw/board/gn...
4,1,공지,1058,평생교육,[평생교육] 2026년 강남아카데미 출결 안내,1201,2026-03-23,https://www.gangnam.go.kr/office/gnsw/board/gn...
5,1,공지,1026,평생교육,[평생교육] 2026년 주차 안내,1184,2025-12-30,https://www.gangnam.go.kr/office/gnsw/board/gn...
6,1,공지,1014,노인일자리사업,[노인일자리사업] 2026년 노인일자리사업 참여자 모집,3106,2025-11-19,https://www.gangnam.go.kr/office/gnsw/board/gn...
7,1,675,1130,건강증진,[건강증진] 대사증후군 예방교육(고지혈증),426,2026-08-25,https://www.gangnam.go.kr/office/gnsw/board/gn...
8,1,674,1131,시니어종합상담사업,[시니어종합상담사업] 마음이음 동년배상담 참여자 모집,321,2026-08-20,https://www.gangnam.go.kr/office/gnsw/board/gn...
9,1,673,1129,평생교육,[평생교육] 나는 자연인이다 2기 참여자 모집,522,2026-08-19,https://www.gangnam.go.kr/office/gnsw/board/gn...


## 4. 프로그램 후보 확인

In [5]:
from src.data_collection_pipeline.detail import is_program_candidate

candidate_df = list_df[
    list_df.apply(
        lambda row: is_program_candidate(
            str(row["title"]),
            str(row["category"]),
        ),
        axis=1,
    )
].copy()

print("프로그램 후보:", len(candidate_df))
display(candidate_df[["category", "title", "post_date", "detail_url"]].head(20))

프로그램 후보: 11


,category,title,post_date,detail_url
6,노인일자리사업,[노인일자리사업] 2026년 노인일자리사업 참여자 모집,2025-11-19,https://www.gangnam.go.kr/office/gnsw/board/gn...
7,건강증진,[건강증진] 대사증후군 예방교육(고지혈증),2026-08-25,https://www.gangnam.go.kr/office/gnsw/board/gn...
8,시니어종합상담사업,[시니어종합상담사업] 마음이음 동년배상담 참여자 모집,2026-08-20,https://www.gangnam.go.kr/office/gnsw/board/gn...
9,평생교육,[평생교육] 나는 자연인이다 2기 참여자 모집,2026-08-19,https://www.gangnam.go.kr/office/gnsw/board/gn...
10,지역복지활성화사업,[지역복지활성화사업] 경로당 프로그램 강사 모집 공고,2026-08-18,https://www.gangnam.go.kr/office/gnsw/board/gn...
11,평생교육,[평생교육] 8~9월 교양강좌 안내,2026-08-13,https://www.gangnam.go.kr/office/gnsw/board/gn...
12,건강증진,[건강증진] 구강건강을 위한 임플란트 관리 교육,2026-08-11,https://www.gangnam.go.kr/office/gnsw/board/gn...
13,노인자원봉사육성사업,[노인자원봉사육성사업] 청년X선배시민이 함께하는 [약속: 복약 안전 프로그램] 참여...,2026-08-04,https://www.gangnam.go.kr/office/gnsw/board/gn...
14,평생교육,[평생교육] 디지털배움터 참여 안내,2026-08-03,https://www.gangnam.go.kr/office/gnsw/board/gn...
15,메타스페이스,[메타스페이스] 8월 해피위너 참여자 모집,2026-07-29,https://www.gangnam.go.kr/office/gnsw/board/gn...


## 5. 상세페이지 수집 및 필드 추출

In [6]:
from src.data_collection_pipeline.detail import run_detail_collection

detail_csv = run_detail_collection(
    list_csv,
    incremental_urls=set(),
)

detail_df = pd.read_csv(
    detail_csv,
    encoding="utf-8-sig",
)

print("상세 성공 건수:", len(detail_df))

cols = [
    "title",
    "recruit_period",
    "apply_method",
    "target",
    "contact",
    "program_period",
    "location",
    "capacity",
    "fee",
]
display(detail_df[[c for c in cols if c in detail_df.columns]].head(10))

2026-08-27 16:11:02,022 | INFO | src.data_collection_pipeline.detail | 상세 수집 성공 | id=1014 | [노인일자리사업] 2026년 노인일자리사업 참여자 모집
2026-08-27 16:11:02,696 | INFO | src.data_collection_pipeline.detail | 상세 수집 성공 | id=1130 | [건강증진] 대사증후군 예방교육(고지혈증)
2026-08-27 16:11:03,400 | INFO | src.data_collection_pipeline.detail | 상세 수집 성공 | id=1131 | [시니어종합상담사업] 마음이음 동년배상담 참여자 모집
2026-08-27 16:11:04,066 | INFO | src.data_collection_pipeline.detail | 상세 수집 성공 | id=1129 | [평생교육] 나는 자연인이다 2기 참여자 모집
2026-08-27 16:11:04,734 | INFO | src.data_collection_pipeline.detail | 상세 수집 성공 | id=1128 | [지역복지활성화사업] 경로당 프로그램 강사 모집 공고
2026-08-27 16:11:05,403 | INFO | src.data_collection_pipeline.detail | 상세 수집 성공 | id=1127 | [평생교육] 8~9월 교양강좌 안내
2026-08-27 16:11:06,109 | INFO | src.data_collection_pipeline.detail | 상세 수집 성공 | id=1126 | [건강증진] 구강건강을 위한 임플란트 관리 교육
2026-08-27 16:11:06,788 | INFO | src.data_collection_pipeline.detail | 상세 수집 성공 | id=1125 | [노인자원봉사육성사업] 청년X선배시민이 함께하는 [약속: 복약 안전 프로그램] 참여자 모집
2026-08-27 16:11:07,451 |

상세 성공 건수: 11


,title,recruit_period,apply_method,target,contact,program_period,location,capacity,fee
0,[노인일자리사업] 2026년 노인일자리사업 참여자 모집,NaN,NaN,NaN,NaN,NaN,강남노인종합복지관 4층 사무실(9:00~17:00),NaN,NaN
1,[건강증진] 대사증후군 예방교육(고지혈증),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,[시니어종합상담사업] 마음이음 동년배상담 참여자 모집,2026. 8. 20. (목) ~ 9. 4. (금),NaN,강남구 지역 내 어르신,양은정 사회복지사(070-8031-2376),NaN,NaN,NaN,NaN
3,[평생교육] 나는 자연인이다 2기 참여자 모집,2026. 8. 19.(수) ~ 선착순 마감,복지관 방문 신청 또는 유선연락,NaN,"평생교육팀(02-549-7070, 박예림 사회복지사)",NaN,NaN,NaN,NaN
4,[지역복지활성화사업] 경로당 프로그램 강사 모집 공고,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,[평생교육] 8~9월 교양강좌 안내,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,[건강증진] 구강건강을 위한 임플란트 관리 교육,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,[노인자원봉사육성사업] 청년X선배시민이 함께하는 [약속: 복약 안전 프로그램] 참여...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,[평생교육] 디지털배움터 참여 안내,8. 4.(화) ~ 8. 6.(목) 오후 6시까지,유선전화 혹은 사무실 내방 접수(추첨),"복지관 회원 20명(정보화 프로그램 수강회원 접수 불가, 26년 정보화 미수강회원 ...","평생교육 담당자, 02-549-7070",NaN,NaN,"20명(정보화 프로그램 수강회원 접수 불가, 26년 정보화 미수강회원 우선선발)",NaN
9,[메타스페이스] 8월 해피위너 참여자 모집,NaN,NaN,NaN,박중연 사회복지사(02-549-7070),NaN,5층 메타스페이스 라,NaN,NaN


## 6. 전처리와 품질 검증

In [7]:
from src.data_collection_pipeline.preprocess import run_preprocess

processed_csv = run_preprocess(detail_csv)

processed_df = pd.read_csv(
    processed_csv,
    encoding="utf-8-sig",
)

print("전처리 결과:", len(processed_df))
display(processed_df.head())

2026-08-27 16:11:09,384 | INFO | src.data_collection_pipeline.preprocess | 품질검증 | rows=11 | duplicate=0 | empty_title=0 | invalid_date=0 | detail_success_rate=100.0%
2026-08-27 16:11:09,387 | INFO | src.data_collection_pipeline.preprocess | 전처리 완료 | 입력=11 | 중복제거=0 | 최종=11 | D:\AI\data_analytics\crawling\gangnam_senior_program_pipeline_v2.0\gangnam_senior_program_pipeline_pro\data\processed\senior_programs_20260827_161049.csv


전처리 결과: 11


,center_name,region,source_page,notice_no,notice_id,category,title,views,post_date,detail_url,...,contact,program_period,location,capacity,fee,phone,body_text,detail_status,capacity_num,program_key
0,강남노인종합복지관,서울특별시 강남구,1,공지,1014,노인일자리사업,[노인일자리사업] 2026년 노인일자리사업 참여자 모집,3106,2025-11-19,https://www.gangnam.go.kr/office/gnsw/board/gn...,...,NaN,NaN,강남노인종합복지관 4층 사무실(9:00~17:00),NaN,NaN,070-8031-2439,[노인일자리사업] 2026년 노인일자리사업 참여자 모집 2025-11-19 조회수 ...,success,NaN,gangnam:1014
1,강남노인종합복지관,서울특별시 강남구,1,675,1130,건강증진,[건강증진] 대사증후군 예방교육(고지혈증),426,2026-08-25,https://www.gangnam.go.kr/office/gnsw/board/gn...,...,NaN,NaN,NaN,NaN,NaN,NaN,[건강증진] 대사증후군 예방교육(고지혈증) 2026-08-25 조회수 426 목록 ...,success,NaN,gangnam:1130
2,강남노인종합복지관,서울특별시 강남구,1,674,1131,시니어종합상담사업,[시니어종합상담사업] 마음이음 동년배상담 참여자 모집,321,2026-08-20,https://www.gangnam.go.kr/office/gnsw/board/gn...,...,양은정 사회복지사(070-8031-2376),NaN,NaN,NaN,NaN,070-8031-2376,[시니어종합상담사업] 마음이음 동년배상담 참여자 모집 2026-08-20 조회수 3...,success,NaN,gangnam:1131
3,강남노인종합복지관,서울특별시 강남구,1,673,1129,평생교육,[평생교육] 나는 자연인이다 2기 참여자 모집,522,2026-08-19,https://www.gangnam.go.kr/office/gnsw/board/gn...,...,"평생교육팀(02-549-7070, 박예림 사회복지사)",NaN,NaN,NaN,NaN,02-549-7070,[평생교육] 나는 자연인이다 2기 참여자 모집 2026-08-19 조회수 522 [...,success,NaN,gangnam:1129
4,강남노인종합복지관,서울특별시 강남구,1,672,1128,지역복지활성화사업,[지역복지활성화사업] 경로당 프로그램 강사 모집 공고,158,2026-08-18,https://www.gangnam.go.kr/office/gnsw/board/gn...,...,NaN,NaN,NaN,NaN,NaN,NaN,[지역복지활성화사업] 경로당 프로그램 강사 모집 공고 2026-08-18 조회수 1...,success,NaN,gangnam:1128


## 7. 전체 15페이지 운영 파이프라인 실행

In [8]:
from src.data_collection_pipeline import run_pipeline

result = run_pipeline(
    incremental=True,
    start_page=1,
    end_page=15,
)

result

2026-08-27 16:11:16,319 | INFO | src.data_collection_pipeline.pipeline | 파이프라인 시작 | batch=20260827_161116 | incremental=True
2026-08-27 16:11:16,328 | INFO | src.data_collection_pipeline.crawling | 목록 수집 시작 | page=1~15 | batch=20260827_161116
2026-08-27 16:11:16,455 | INFO | src.data_collection_pipeline.crawling | 목록 수집 성공 | page=1 | status=200 | bytes=142264 | gangnam_notice_page_001.html
2026-08-27 16:11:17,047 | INFO | src.data_collection_pipeline.crawling | 목록 수집 성공 | page=2 | status=200 | bytes=142264 | gangnam_notice_page_002.html
2026-08-27 16:11:17,639 | INFO | src.data_collection_pipeline.crawling | 목록 수집 성공 | page=3 | status=200 | bytes=142264 | gangnam_notice_page_003.html
2026-08-27 16:11:18,251 | INFO | src.data_collection_pipeline.crawling | 목록 수집 성공 | page=4 | status=200 | bytes=142264 | gangnam_notice_page_004.html
2026-08-27 16:11:18,857 | INFO | src.data_collection_pipeline.crawling | 목록 수집 성공 | page=5 | status=200 | bytes=142264 | gangnam_notice_page_005.html
2026-08

PipelineResult(batch_id='20260827_161116', list_count=17, detail_count=11, processed_count=11, loaded_count=0, processed_csv=WindowsPath('D:/AI/data_analytics/crawling/gangnam_senior_program_pipeline_v2.0/gangnam_senior_program_pipeline_pro/data/processed/senior_programs_20260827_161116.csv'))

## 8. 로그 확인

실제 운영에서 가장 먼저 확인할 파일입니다.

`logs/pipeline_YYYYMMDD.log`

여기서 페이지별 수집 성공/실패, 상세페이지 실패 URL, 전처리 건수, DB 적재 결과를 확인할 수 있습니다.